# ⚡ GIADA roadmap Task 7 — sequenze di voltage step
Questa è la Task 7 originale della roadmap. I notebook storici 07/07b/07c riguardavano invece Ca-HVA closed-loop. La Task 6 ha già coperto step singoli e bifasici; qui variamo storia temporale a parità di estremi e stato iniziale.


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_roadmap_task7');GIADA_REPO=WORK/'giada';TEACHER_REPO=WORK/'neuron_as_deep_net'
assert not WORK.exists(),'Usa una sessione Kaggle nuova: directory Task 7 già presente.'
WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip();print({'revision':REVISION})


## 📁 Unico input aggiuntivo
Serve l'artefatto completo e immutabile `giada_primitive_scaling_laws.zip` della Task 5, come nella Task 6. Non servono il vecchio ZIP della Task 6 né il dataset targeted v1.1.


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
from src.giada_teacher import ExtractedGateFormula,compile_nmodl
from src.giada_teacher.voltage_path_stress import VoltagePathStressConfig,verified_task5_root,EXPECTED_TASK5_ARCHIVE_SHA256,EXPECTED_TASK5_REPORT_SHA256
from src.giada_teacher.voltage_step_sequences import run_roadmap_task7,FAMILIES
import torch
assert torch.cuda.is_available(),'Serve GPU CUDA per i candidati congelati.'
def sha(path):
 d=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''):d.update(chunk)
 return d.hexdigest()
override=os.environ.get('GIADA_TASK5_ARTIFACT');INPUT_ROOT=Path('/kaggle/input')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
 candidates+=list(INPUT_ROOT.rglob('giada_primitive_scaling_laws.zip'))
 candidates += [p for p in INPUT_ROOT.rglob('archive.zip') if 'scaling' in str(p).lower()]
 candidates += [p.parent for p in INPUT_ROOT.rglob('frozen_scaling_checkpoints.pt')]
def exact(p):
 try:return sha(p)==EXPECTED_TASK5_ARCHIVE_SHA256 if p.is_file() else sha(p/'final_report.json')==EXPECTED_TASK5_REPORT_SHA256
 except Exception:return False
TASK5_SOURCE=next((p.resolve() for p in candidates if p.exists() and exact(p)),None)
assert TASK5_SOURCE is not None,'Aggiungi giada_primitive_scaling_laws.zip esatto agli input o imposta GIADA_TASK5_ARTIFACT.'
TASK5_ROOT=verified_task5_root(TASK5_SOURCE,WORK/'task5_verified')
display({'task5_verified':str(TASK5_ROOT),'families':FAMILIES,'gpu':torch.cuda.get_device_name(0)})


In [ ]:
if shutil.which('nrnivmodl') is None:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 os.environ['PATH']=str(Path(sys.executable).parent)+os.pathsep+os.environ.get('PATH','')
assert shutil.which('nrnivmodl') is not None and shutil.which('gcc') is not None,'NEURON/gcc mancanti.'
MOD=TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod'
formula=ExtractedGateFormula.from_mod(MOD)
mechanism_root=compile_nmodl(MOD,WORK/'compiled_mechanism')
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_roadmap_task7_voltage_step_sequences')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Usa una sessione nuova.'
config=VoltagePathStressConfig()
display({'neuron_mechanism':str(mechanism_root),'development_per_family':config.development_per_family,'sealed_per_family':config.sealed_per_family})


## 🔬 Pilot, freeze e sealed nello stesso run
Il pilot NEURON precede il test. Development e sealed usano percorsi distinti; nessun candidato viene riaddestrato.


In [ ]:
report=run_roadmap_task7(formula,TASK5_ROOT,mechanism_root,OUTPUT_DIR,config,code_revision=REVISION)
display({'valid':report['valid'],'registered_gate_passed':report['registered_gate_passed'],'pilot':report['oracle_pilot'],'sealed_overlap':report['sealed_overlap_count'],'lut_scores':report['lut_known_path_scores'],'paired_teacher_differences':report['sealed_pairs'],'exogenous_only':report['teacher_voltage_path_is_exogenous']})
assert report['valid'] and report['sealed_overlap_count']==0 and not report['selection_used_sealed']


## 📦 Scarica i soli report
Metodo Blob/base64 concordato; i checkpoint Task 5 non vengono ri-zippati.


In [ ]:
EXPORT=Path('/kaggle/working/giada_roadmap_task7_report');assert not EXPORT.exists()
EXPORT.mkdir()
for name in ('development_report.json','selection_freeze.json','final_report.json'):
 shutil.copy2(OUTPUT_DIR/name,EXPORT/name)
archive=Path(shutil.make_archive('/kaggle/working/giada_roadmap_task7_report','zip',EXPORT.parent,EXPORT.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
